In [0]:
%run ../00-common/01.environment-config

In [0]:
%python
control_table = f"{catalog_name}.{control_schema}.batch_control"

In [0]:
%python
from pyspark.sql import functions as F

#Pega as pastas de batch da landing
landing_batches = sorted([
    file.name.rstrip("/")
    for file in dbutils.fs.ls(landing_folder_path)
    if file.isDir()
])

#Lê os batches já rastreados
if spark.catalog.tableExists(control_table):
    tracked_batches = [
        row.batch_id
        for row in (
            spark.table(control_table)
            .filter(F.col("status").isin("in_progress", "completed"))
            .select("batch_id")
            .distinct()
            .collect()
        )
    ]
else:
    tracked_batches = []

#Identifica o batch mais antigo ainda não processado
new_batches = sorted(list(set(landing_batches) - set(tracked_batches)))
next_batch = new_batches[0] if new_batches else None

print(f"Landing batches      : {landing_batches}")
print(f"Tracked batches      : {tracked_batches}")
print(f"Next batch to process: {next_batch}")

if next_batch is None:
    dbutils.jobs.taskValues.set(key="p_batch_id", value="")
    dbutils.jobs.taskValues.set(key="has_batch", value="false")
else:
    dbutils.jobs.taskValues.set(key="p_batch_id", value=next_batch)
    dbutils.jobs.taskValues.set(key="has_batch", value="true")

Landing batches      : ['2025-01']
Tracked batches      : []
Next batch to process: 2025-01
